## Addresses issue with some river reaches being unable to produce a curve fit to the reworking area growth and or overlap decay curves

In [2]:
import pandas as pd
import os
from pathlib import Path

def check_reach_diagnostics(input_csv_path, output_csv_path):
    """
    Check mask counts and mobility CSV existence for each reach in the dataset.
    
    Parameters:
        input_csv_path (str): Path to the Greenberg et al. 2024 river datasheet CSV
        output_csv_path (str): Path to save the diagnostic output CSV
    
    Outputs:
        CSV file with columns:
            - river_name: Name of the river
            - ds_order: Reach number
            - num_masks: Count of .tif files in the Cleaned folder
            - mobility_csv_exists: 'yes' or 'no' indicating if mobility metrics CSV exists
    """
    # Read the input CSV
    river_data = pd.read_csv(input_csv_path)
    
    # List to store diagnostic results
    diagnostics = []
    
    # Iterate through each river
    for _, row in river_data.iterrows():
        river_name = row['river_name']
        working_directory = row['working_directory']
        
        print(f"Checking {river_name}...")
        
        # Define paths
        base_raster_dir = os.path.join(working_directory, "RiverMapping", "RiverMasks", river_name)
        mobility_csv_path = os.path.join(working_directory, "RiverMapping", "Mobility", river_name, 
                                        f"{river_name}_mobility_metrics.csv")
        
        # Check if mobility CSV exists (once per river, applies to all reaches)
        mobility_exists = "yes" if os.path.exists(mobility_csv_path) else "no"
        
        # Check if base raster directory exists
        if not os.path.exists(base_raster_dir):
            print(f"  Warning: Raster directory not found for {river_name}")
            diagnostics.append({
                'river_name': river_name,
                'ds_order': 'N/A',
                'num_masks': 0,
                'mobility_csv_exists': mobility_exists
            })
            continue
        
        # Find all reach directories
        try:
            reach_dirs = [d for d in os.listdir(base_raster_dir) 
                         if d.startswith("reach_") and os.path.isdir(os.path.join(base_raster_dir, d))]
            
            if not reach_dirs:
                print(f"  Warning: No reach directories found for {river_name}")
                diagnostics.append({
                    'river_name': river_name,
                    'ds_order': 'N/A',
                    'num_masks': 0,
                    'mobility_csv_exists': mobility_exists
                })
                continue
            
            # Sort reaches numerically
            reach_numbers = sorted([int(d.split('_')[1]) for d in reach_dirs])
            
            # Process each reach
            for ds_order in reach_numbers:
                cleaned_dir = os.path.join(base_raster_dir, f"reach_{ds_order}", "Cleaned")
                
                # Count .tif files
                if os.path.exists(cleaned_dir):
                    tif_files = [f for f in os.listdir(cleaned_dir) if f.endswith('.tif')]
                    num_masks = len(tif_files)
                else:
                    num_masks = 0
                    print(f"  Warning: Cleaned directory not found for reach {ds_order}")
                
                # Add to diagnostics
                diagnostics.append({
                    'river_name': river_name,
                    'ds_order': ds_order,
                    'num_masks': num_masks,
                    'mobility_csv_exists': mobility_exists
                })
                
                print(f"  Reach {ds_order}: {num_masks} masks, Mobility CSV: {mobility_exists}")
        
        except Exception as e:
            print(f"  Error processing {river_name}: {e}")
            diagnostics.append({
                'river_name': river_name,
                'ds_order': 'ERROR',
                'num_masks': 0,
                'mobility_csv_exists': mobility_exists
            })
    
    # Create DataFrame and save to CSV
    diagnostics_df = pd.DataFrame(diagnostics)
    diagnostics_df.to_csv(output_csv_path, index=False)
    
    print(f"\nDiagnostic results saved to {output_csv_path}")
    print(f"Total reaches checked: {len(diagnostics_df)}")
    print(f"Reaches with masks: {len(diagnostics_df[diagnostics_df['num_masks'] > 0])}")
    print(f"Rivers with mobility CSV: {diagnostics_df['mobility_csv_exists'].value_counts().get('yes', 0)} yes, {diagnostics_df['mobility_csv_exists'].value_counts().get('no', 0)} no")
    
    return diagnostics_df


# Example usage
if __name__ == "__main__":
    input_csv = r"E:\Dissertation\Data\Zhaoetal2025_river_datasheet.csv"
    output_csv = r"C:\Users\huckr\Desktop\UCSB\Dissertation\Data\Code\Troubleshooting\Zhao_reach_diagnostics.csv"
    
    results = check_reach_diagnostics(input_csv, output_csv)
    
    # Display summary statistics
    print("\n=== SUMMARY STATISTICS ===")
    print(f"Mean masks per reach: {results['num_masks'].mean():.2f}")
    print(f"Median masks per reach: {results['num_masks'].median():.0f}")
    print(f"Min masks: {results['num_masks'].min()}")
    print(f"Max masks: {results['num_masks'].max()}")
    print(f"\nReaches with < 5 masks:")
    low_mask_reaches = results[results['num_masks'] < 5]
    if len(low_mask_reaches) > 0:
        for _, row in low_mask_reaches.iterrows():
            print(f"  {row['river_name']} Reach {row['ds_order']}: {row['num_masks']} masks")
    else:
        print("  None")

Checking Aladan_VerkhoyanskiyPerevoz...
  Reach 1: 18 masks, Mobility CSV: yes
Checking Amazonas_Jatuarana...
  Reach 1: 36 masks, Mobility CSV: yes
Checking Amur_Komsomolsk...
  Reach 1: 37 masks, Mobility CSV: yes
Checking Benue_Umaisha...
  Reach 1: 16 masks, Mobility CSV: no
Checking BolshayaKet_Rodyonovka...
  Reach 1: 21 masks, Mobility CSV: yes
Checking Brahmaputra_Pasighat...
  Reach 1: 37 masks, Mobility CSV: yes
Checking Chari_Bousso...
  Reach 1: 18 masks, Mobility CSV: yes
Checking Chari_Guelengdeng...
  Reach 1: 17 masks, Mobility CSV: yes
Checking Chari_Ndjamena...
  Reach 1: 18 masks, Mobility CSV: no
Checking Chari_Sahr...
  Reach 1: 18 masks, Mobility CSV: no
Checking Fraser_Hope...
  Reach 1: 38 masks, Mobility CSV: yes
Checking Gandak_Devghat...
  Reach 1: 33 masks, Mobility CSV: yes
Checking Helmand_Kajaki...
  Reach 1: 31 masks, Mobility CSV: no
Checking Helmand_Malakhan...
  Reach 1: 26 masks, Mobility CSV: no
Checking Indus_Attock...
  Reach 1: 33 masks, Mobility